In [ ]:
import os
from sklearn.model_selection import KFold
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import optuna


input_path = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\AND_result"
label_path = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level2\ulcer_images"

def load_images(path):
    images = []
    for filename in sorted(os.listdir(path)):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            img = Image.open(os.path.join(path, filename)).resize((256, 256))
            images.append(np.array(img))
    return np.array(images)

input_images = load_images(input_path)
label_images = load_images(label_path)

assert len(input_images) == len(label_images), "Mismatch between input and label counts"

k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

train_test_splits = []
for train_idx, test_idx in kf.split(input_images):
    X_train, X_test = input_images[train_idx], input_images[test_idx]
    y_train, y_test = label_images[train_idx], label_images[test_idx]
    train_test_splits.append((X_train, X_test, y_train, y_test))

In [ ]:

# ----------------------- Dataset -----------------------
class CorneaDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]

        # To tensor and normalize
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).unsqueeze(0).float() / 255.0

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0.5).float()
        return image, mask

# ---------------------- Metrics -----------------------
def specificity_score(pred, target, threshold=0.5, epsilon=1e-6):
    """
    Compute specificity = TN / (TN + FP) for a batch,
    using a custom threshold after sigmoid.
    pred: logits tensor, shape [B, 1, H, W]
    target: binary mask tensor, same shape
    """
    probs = torch.sigmoid(pred)
    preds = (probs > threshold).float()
    tn = ((1 - preds) * (1 - target)).sum(dim=(1, 2, 3))
    fp = (preds * (1 - target)).sum(dim=(1, 2, 3)) + epsilon
    spec = tn / (tn + fp)
    return spec.mean()


def eval_specificity_threshold(model, loader, threshold, device):
    """
    Evaluate model on loader and return average specificity
    at given threshold.
    """
    model.eval()
    spec_sum, batches = 0.0, 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            spec = specificity_score(outputs, masks, threshold)
            spec_sum += spec.item()
            batches += 1
    return spec_sum / batches

# -------------------- AttentionUNet -------------------
class AttentionUNet(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=1,
        features=[16, 32, 64, 128],
        kernel_size=3,
        dropout=0.0
    ):
        super(AttentionUNet, self).__init__()
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.dropout_rate = dropout

        # Encoder
        self.encoder1 = self.conv_block(in_channels, features[0])
        self.pool1 = nn.MaxPool2d(2, 2)
        self.encoder2 = self.conv_block(features[0], features[1])
        self.pool2 = nn.MaxPool2d(2, 2)
        self.encoder3 = self.conv_block(features[1], features[2])
        self.pool3 = nn.MaxPool2d(2, 2)
        self.encoder4 = self.conv_block(features[2], features[3])
        self.pool4 = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = self.conv_block(features[3], features[3] * 2)

        # Decoder with attention
        self.up4 = nn.ConvTranspose2d(features[3] * 2, features[3], kernel_size=2, stride=2)
        self.att4 = AttentionBlock(F_g=features[3], F_l=features[3], F_int=features[3] // 2)
        self.decoder4 = self.conv_block(features[3] * 2, features[3])

        self.up3 = nn.ConvTranspose2d(features[3], features[2], kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=features[2], F_l=features[2], F_int=features[2] // 2)
        self.decoder3 = self.conv_block(features[2] * 2, features[2])

        self.up2 = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=features[1], F_l=features[1], F_int=features[1] // 2)
        self.decoder2 = self.conv_block(features[1] * 2, features[1])

        self.up1 = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=features[0], F_l=features[0], F_int=features[0] // 2)
        self.decoder1 = self.conv_block(features[0] * 2, features[0])

        # Final layer
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_ch, out_ch):
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=self.kernel_size, padding=self.padding),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        ]
        if self.dropout_rate > 0:
            layers.append(nn.Dropout(self.dropout_rate))
        layers += [
            nn.Conv2d(out_ch, out_ch, kernel_size=self.kernel_size, padding=self.padding),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        ]
        if self.dropout_rate > 0:
            layers.append(nn.Dropout(self.dropout_rate))
        return nn.Sequential(*layers)

    def forward(self, x):
        e1 = self.encoder1(x); p1 = self.pool1(e1)
        e2 = self.encoder2(p1); p2 = self.pool2(e2)
        e3 = self.encoder3(p2); p3 = self.pool3(e3)
        e4 = self.encoder4(p3); p4 = self.pool4(e4)

        b = self.bottleneck(p4)

        up4 = self.up4(b)
        att4 = self.att4(g=up4, x=e4)
        d4 = self.decoder4(torch.cat([up4, att4], dim=1))

        up3 = self.up3(d4)
        att3 = self.att3(g=up3, x=e3)
        d3 = self.decoder3(torch.cat([up3, att3], dim=1))

        up2 = self.up2(d3)
        att2 = self.att2(g=up2, x=e2)
        d2 = self.decoder2(torch.cat([up2, att2], dim=1))

        up1 = self.up1(d2)
        att1 = self.att1(g=up1, x=e1)
        d1 = self.decoder1(torch.cat([up1, att1], dim=1))

        return self.final_conv(d1)

# -------------------- Optuna Objective -------------------
def objective(trial):
    # Hyperparameters
    base = trial.suggest_categorical("base", [16, 32, 64])
    multipliers = [1, 2, 4, 8]
    features = [base * m for m in multipliers]
    kernel_size = trial.suggest_int("kernel_size", 3, 7, step=2)
    dropout = trial.suggest_categorical("dropout", [0.0, 0.1, 0.2])
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-3)
    weight_decay = trial.suggest_loguniform("wd", 1e-6, 1e-3)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    threshold = trial.suggest_uniform("threshold", 0.4, 0.9)

    # Data loaders (assumes train_dataset, val_dataset defined globally)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)

    # Model, optimizer, scheduler
    model = AttentionUNet(in_channels=3, out_channels=1,
                          features=features,
                          kernel_size=kernel_size,
                          dropout=dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

    # Brief training loop
    criterion = nn.BCEWithLogitsLoss()
    for epoch in range(10):
        model.train()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward(); optimizer.step()
        scheduler.step(loss)

    # Evaluate specificity
    spec = eval_specificity_threshold(model, val_loader, threshold, device)
    return spec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)
print("Best hyperparameters:", study.best_trial.params)